# Teste de acesso: Steam Web API e SteamSpy API

Notebook de validacao da PJBL (Data Analytics - tema Steam). Testa se da pra puxar dados ao vivo
direto da **Steam Web API** e da **SteamSpy API**, ambas publicas e sem necessidade de API key,
pra usar como fonte complementar aos datasets estaticos do Kaggle.

In [1]:
import requests

# Steam Web API - endpoint publico, sem key.
# ISteamApps/GetAppList/v2 e o endpoint classicamente documentado pra listar
# todos os apps da Steam, mas testado ao vivo (04/09/2026) retorna 404 -
# a Valve parece ter removido esse metodo do ISteamApps (confirmado via
# ISteamWebAPIUtil/GetSupportedAPIList, que nao lista mais GetAppList ali).
# Fica registrado como achado real do teste, nao como erro de codigo.
url_applist = "https://api.steampowered.com/ISteamApps/GetAppList/v2/"
resp = requests.get(url_applist, timeout=15)
print(f"GetAppList/v2 - status: {resp.status_code} (esperado 404 nesse teste)")

# Substituto que funciona sem key: ISteamNews/GetNewsForApp, noticias de um
# jogo especifico (appid 570 = Dota 2). Confirma que a Steam Web API publica
# esta acessivel de fato.
url_news = "https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/"
resp = requests.get(url_news, params={"appid": 570, "count": 3, "maxlength": 150}, timeout=15)
resp.raise_for_status()
news = resp.json()["appnews"]["newsitems"]

print(f"\nGetNewsForApp - status: {resp.status_code}")
print(f"Noticias retornadas: {len(news)}")
for n in news:
    print("-", n["title"])


GetAppList/v2 - status: 404 (esperado 404 nesse teste)

GetNewsForApp - status: 200
Noticias retornadas: 3
- The International 2026 Grand Champions
- 'Dude no joke looks like Hugh Jackman now': The internet is swooning over Gabe Newell's svelte new look at The International
- The International Main Event


In [2]:
# Steam Store API - detalhes de um jogo especifico (sem key).
# appid 570 = Dota 2.
appid = 570
url_details = f"https://store.steampowered.com/api/appdetails?appids={appid}"
resp = requests.get(url_details, timeout=15)
resp.raise_for_status()
data = resp.json()[str(appid)]

print(f"Status: {resp.status_code}")
print(f"success: {data['success']}")
if data["success"]:
    game = data["data"]
    print(f"Nome: {game['name']}")
    print(f"Preco: {game.get('price_overview', {}).get('final_formatted', 'Free/N-A')}")
    print(f"Generos: {[g['description'] for g in game.get('genres', [])]}")
    print(f"Data de lancamento: {game.get('release_date', {}).get('date')}")


Status: 200
success: True
Nome: Dota 2
Preco: Free/N-A
Generos: ['Action', 'Strategy', 'Free To Play']
Data de lancamento: 9 Jul, 2013


In [3]:
# SteamSpy API - dados agregados de jogadores (owners estimados, peak CCU etc).
# Documentacao: https://steamspy.com/api.php
url_spy = "https://steamspy.com/api.php?request=appdetails&appid=570"
resp = requests.get(url_spy, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
data = resp.json()

print(f"Status: {resp.status_code}")
print(f"Nome: {data.get('name')}")
print(f"Owners (estimado): {data.get('owners')}")
print(f"Peak CCU (24h): {data.get('ccu')}")
print(f"Reviews positivas/negativas: {data.get('positive')}/{data.get('negative')}")


Status: 200
Nome: Dota 2
Owners (estimado): 100,000,000 .. 200,000,000
Peak CCU (24h): 623941
Reviews positivas/negativas: 2037143/461826
